# 통신데이터 검증

역할: parquet 후보가 저장만 된 상태가 아니라 바로 분석에 재사용 가능한 상태인지 확인한다.

검증 기준:

- 다시 불러오기 되는지
- 총 행 수가 원본 후보 parquet와 맞는지
- 기간 범위가 정상인지
- 핵심 컬럼 타입이 의도대로 들어갔는지
- 코드값을 확정 / 추정 / 미확인으로 관리했는지

## 전체 t 데이터 검증 범위

아래 검증은 네가 실제로 확인한 T4~T27 전체 파일을 대상으로 파일 존재 여부, 재불러오기, 행 수, 기간 범위를 점검한다.  
그 다음 분석 후보 대상인 T13/T25/T26/T27은 파일이 커서 별도로 더 엄격하게 검증한다.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

DATA_DIR = Path('../data')
con = duckdb.connect()

ALL_T_FILES = {
    'T4':  {'file': 't4_2023_2025_all_date_final.parquet',  'format': 'parquet'},
    'T5':  {'file': 't5_2023_2025_all_date_final.parquet',  'format': 'parquet'},
    'T6':  {'file': 't6_2023_2025_all_date_final.parquet',  'format': 'parquet'},
    'T7':  {'file': 't7_2023_2025_all_date_final.parquet',  'format': 'parquet'},
    'T8':  {'file': 't8_2023_2025_all_date_final.parquet',  'format': 'parquet'},
    'T9':  {'file': 't9_2023_2025_all_date_final.parquet',  'format': 'parquet'},
    'T10': {'file': 't10_2023_2025_all_date_final.parquet', 'format': 'parquet'},
    'T11': {'file': 't11_2023_2025_all_date_final.parquet', 'format': 'parquet'},
    'T12': {'file': 't12_2023_2025_all_final_v2.parquet',   'format': 'parquet'},
    'T13': {'file': 't13_seongnam_final.parquet',           'format': 'parquet'},
    'T14': {'file': 't14_2023_2025_all_final_v2.parquet',   'format': 'parquet'},
    'T16': {'file': 't16_2023_2025_all_date_final.parquet', 'format': 'parquet'},
    'T20': {'file': 't20_2023_2025_all_date_final.csv',     'format': 'csv'},
    'T21': {'file': 't21_2023_2025_all_date_final.csv',     'format': 'csv'},
    'T22': {'file': 't22_2023_2025_all_date_final.parquet', 'format': 'parquet'},
    'T23': {'file': 't23_2023_2025_all_date_final.parquet', 'format': 'parquet'},
    'T24': {'file': 't24_seongnam_final.parquet', 'format': 'parquet'},
    'T25': {'file': 't25_seongnam_final.parquet',           'format': 'parquet'},
    'T26': {'file': 't26_seongnam_final.parquet',           'format': 'parquet'},
    'T27': {'file': 't27_seongnam_final.parquet',           'format': 'parquet'},
}

def table_stats(table, info):
    path = DATA_DIR / info['file']
    if not path.exists():
        return {'table': table, 'file': info['file'], 'exists': False, 'reload_ok': False}

    p = str(path).replace('\\', '/')
    if info['format'] == 'parquet':
        schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{p}')").fetchdf()
        cols = set(schema['column_name'])
        date_col = 'ETL_YMD' if 'ETL_YMD' in cols else None
        if date_col:
            row_count, min_ymd, max_ymd = con.execute(f"""
                SELECT COUNT(*), CAST(MIN({date_col}) AS DATE), CAST(MAX({date_col}) AS DATE)
                FROM read_parquet('{p}')
            """).fetchone()
        else:
            row_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{p}')").fetchone()[0]
            min_ymd, max_ymd = None, None
        return {
            'table': table,
            'file': info['file'],
            'format': info['format'],
            'exists': True,
            'reload_ok': True,
            'row_count': row_count,
            'min_ymd': min_ymd,
            'max_ymd': max_ymd,
            'column_count': len(cols),
        }

    row_count = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{p}', header=true)").fetchone()[0]
    return {
        'table': table,
        'file': info['file'],
        'format': info['format'],
        'exists': True,
        'reload_ok': True,
        'row_count': row_count,
        'min_ymd': None,
        'max_ymd': None,
        'column_count': None,
    }

all_t_validation = pd.DataFrame([
    table_stats(table, info)
    for table, info in ALL_T_FILES.items()
])
all_t_validation

In [ ]:
all_t_out_path = DATA_DIR / 'telecom_all_t_validation_summary.csv'
all_t_validation.to_csv(all_t_out_path, index=False, encoding='utf-8-sig')
all_t_out_path

In [ ]:
from pathlib import Path
import glob
import duckdb
import pandas as pd

DATA_DIR = Path('../data')
FINAL_FILES = {
    'T13': DATA_DIR / 't13_seongnam_final.parquet',
    'T24': DATA_DIR / 't24_seongnam_final.parquet',
    'T25': DATA_DIR / 't25_seongnam_final.parquet',
    'T26': DATA_DIR / 't26_seongnam_final.parquet',
    'T27': DATA_DIR / 't27_seongnam_final.parquet',
}
SOURCE_FILES = {
    'T13': DATA_DIR / 't13_2023_2025_all_final_v2.parquet',
    'T24': DATA_DIR / 't24_2023_2025_all_date_final.parquet',
    'T25': DATA_DIR / 't25_2023_2025_all_final_v2.parquet',
    'T26': DATA_DIR / 't26_2023_2025_all_final.parquet',
    'T27': DATA_DIR / 't27_2023_2025_all_final.parquet',
}
RAW_PATTERNS = {
    'T13': str(DATA_DIR / 'T13_*.csv'),
    'T25': str(DATA_DIR / 'T25_*.csv'),
    'T26': str(DATA_DIR / 'T26_*.csv'),
    'T27': str(DATA_DIR / 'T27_*.csv'),
}
EXPECTED_MIN = '2023-01-01'
EXPECTED_MAX = '2025-12-31'
CORE_COLUMNS = {
    'T13': ['CNT', 'PURPOSE', 'SEX_CD', 'AGE_GRP'],
    'T24': ['CNT', 'PURPOSE', 'ADMI_CD'],
    'T25': ['CNT', 'D_CTY_CD', 'O_CTY_CD'],
    'T26': ['DURATION', 'PURPOSE'],
    'T27': ['TRANS_GB', 'PURPOSE', 'DURATION'],
}

con = duckdb.connect()

In [ ]:
def parquet_stats(path):
    path = str(path).replace('\\', '/')
    return con.execute(f'''
        SELECT COUNT(*) AS row_count,
               CAST(MIN(ETL_YMD) AS DATE) AS min_ymd,
               CAST(MAX(ETL_YMD) AS DATE) AS max_ymd
        FROM read_parquet('{path}')
    ''').fetchone()


def parquet_schema(path):
    path = str(path).replace('\\', '/')
    return con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").fetchdf()


def raw_csv_row_count(pattern):
    files = sorted(glob.glob(pattern))
    if not files:
        return None, []
    file_sql = ', '.join("'" + f.replace('\\', '/') + "'" for f in files)
    count = con.execute(f'''
        SELECT COUNT(*)
        FROM read_csv_auto([{file_sql}], header=true, union_by_name=true, ignore_errors=true)
    ''').fetchone()[0]
    return count, files

In [ ]:
rows = []
for table, final_path in FINAL_FILES.items():
    source_path = SOURCE_FILES[table]
    final_count, min_ymd, max_ymd = parquet_stats(final_path)
    source_count, _, _ = parquet_stats(source_path)
    raw_count, raw_files = raw_csv_row_count(RAW_PATTERNS[table])
    schema = parquet_schema(final_path)
    col_types = dict(zip(schema['column_name'], schema['column_type']))
    required = CORE_COLUMNS[table]
    existing_required = [c for c in required if c in col_types]
    missing_required = [c for c in required if c not in col_types]

    rows.append({
        'table': table,
        'final_file': final_path.name,
        'reload_ok': True,
        'row_count': final_count,
        'source_parquet_row_count': source_count,
        'source_match': final_count == source_count,
        'raw_csv_files': len(raw_files),
        'raw_csv_row_count': raw_count if raw_count is not None else 'SKIP: raw CSV not found',
        'raw_match': (final_count == raw_count) if raw_count is not None else 'SKIP',
        'min_ymd': str(min_ymd),
        'max_ymd': str(max_ymd),
        'period_ok': str(min_ymd) == EXPECTED_MIN and str(max_ymd) == EXPECTED_MAX,
        'core_columns_present': ', '.join(existing_required),
        'core_columns_missing': ', '.join(missing_required),
        'core_column_types': {c: col_types[c] for c in existing_required},
    })

validation_result = pd.DataFrame(rows)
validation_result

In [ ]:
out_path = DATA_DIR / 'telecom_final_validation_summary.csv'
validation_result.to_csv(out_path, index=False, encoding='utf-8-sig')
out_path

In [ ]:
code_rows = []
for table, path in FINAL_FILES.items():
    schema = parquet_schema(path)
    columns = set(schema['column_name'])
    for col in ['PURPOSE', 'TRANS_GB', 'SEX_CD', 'AGE_GRP']:
        if col not in columns:
            continue
        p = str(path).replace('\\', '/')
        dist = con.execute(f'''
            SELECT CAST({col} AS VARCHAR) AS code_value, COUNT(*) AS row_count
            FROM read_parquet('{p}')
            GROUP BY 1
            ORDER BY 1
        ''').fetchdf()
        for _, r in dist.iterrows():
            status = '미확인'
            meaning = '공식 정의서 확인 필요'
            if col == 'SEX_CD' and r['code_value'] in ['M', 'F']:
                status = '확정'
                meaning = '남성' if r['code_value'] == 'M' else '여성'
            elif col == 'SEX_CD' and r['code_value'] == 'W':
                meaning = '기타/미확인'
            code_rows.append({
                'table': table,
                'column': col,
                'code_value': r['code_value'],
                'row_count': r['row_count'],
                'meaning': meaning,
                'status': status,
            })

code_definition = pd.DataFrame(code_rows)
code_definition

In [ ]:
code_out_path = DATA_DIR / 'telecom_code_definition_status.csv'
code_definition.to_csv(code_out_path, index=False, encoding='utf-8-sig')
code_out_path

## 검증 항목 상세

검증은 단순히 파일이 저장됐는지를 보는 것이 아니라, 바로 EDA와 모델링 후보 변수 생성에 재사용할 수 있는지를 확인하는 과정이다.

| 검증 항목 | 확인 내용 | 실패 시 처리 |
|---|---|---|
| 파일 존재 | 후보 파일이 `data` 폴더에 있는지 | 통합 노트북에서 파일명/경로 재확인 |
| 재불러오기 | parquet/csv가 DuckDB로 다시 읽히는지 | 파일 손상, 포맷, 인코딩 확인 |
| 행 수 | 원본 후보 파일과 후보 파일 행 수가 같은지 | 중복/누락 여부 확인 |
| 기간 범위 | 2023-01-01 ~ 2025-12-31 범위인지 | 날짜 변환 로직 확인 |
| 핵심 컬럼 | CNT, PURPOSE, DURATION 등 핵심 컬럼 존재/타입 | 컬럼명 통일 또는 테이블 제외 판단 |
| 코드값 | PURPOSE, TRANS_GB, SEX_CD, AGE_GRP 범위 | 공식 정의서 확인 전까지 미확인 유지 |

In [ ]:
# 예측 변수 후보 테이블의 결측치 점검용 쿼리 생성
NULL_CHECK_COLUMNS = {
    'T13': ['ETL_YMD', 'CNT', 'PURPOSE', 'SEX_CD', 'AGE_GRP'],
    'T24': ['ETL_YMD', 'CNT', 'PURPOSE', 'ADMI_CD'],
    'T25': ['ETL_YMD', 'CNT', 'PURPOSE', 'TRANS_GB', 'O_CTY_CD', 'D_CTY_CD'],
    'T26': ['ETL_YMD', 'CNT', 'PURPOSE', 'TRANS_GB', 'DURATION'],
    'T27': ['ETL_YMD', 'CNT', 'PURPOSE', 'TRANS_GB', 'DURATION'],
}

null_rows = []
for table, cols in NULL_CHECK_COLUMNS.items():
    path = FINAL_FILES[table]
    schema = parquet_schema(path)
    existing_cols = set(schema['column_name'])
    p = str(path).replace('\\', '/')
    checks = []
    for col in cols:
        if col in existing_cols:
            checks.append(f"SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) AS {col}_nulls")
    if not checks:
        continue
    result = con.execute(f"""
        SELECT '{table}' AS table_name, COUNT(*) AS row_count, {', '.join(checks)}
        FROM read_parquet('{p}')
    """).fetchdf()
    null_rows.append(result)

final_null_check = pd.concat(null_rows, ignore_index=True)
final_null_check

In [ ]:
# 기간 검증을 월 단위로도 확인한다.
monthly_validation_parts = []
for table, path in FINAL_FILES.items():
    p = str(path).replace('\\', '/')
    monthly_validation_parts.append(con.execute(f'''
        SELECT
            '{table}' AS table_name,
            STRFTIME(CAST(ETL_YMD AS DATE), '%Y-%m') AS ym,
            COUNT(*) AS row_count
        FROM read_parquet('{p}')
        GROUP BY 1, 2
        ORDER BY 1, 2
    ''').fetchdf())

final_monthly_validation = pd.concat(monthly_validation_parts, ignore_index=True)
final_monthly_validation

## 검증 결과 해석

- T13/T24/T25/T26/T27은 EDA에서 변수 후보로 연결한 테이블이므로 핵심 컬럼까지 별도로 확인한다.
- T4~T23의 보조 테이블은 전체 구조 파악용으로 파일 존재, 재불러오기, 기간/행 수 중심으로 확인한다.
- `SEX_CD = W`는 공식 정의서 확인 전까지 기타/미확인으로 둔다.
- PURPOSE, TRANS_GB, AGE_GRP는 코드값 분포만 관리하고 의미 해석은 공식 정의서 확인 후 후보로 정리한다.

## 코드값 정의표

공식 정의서를 확인하기 전까지 코드값은 확정/추정/미확인으로 분리한다. 특히 `SEX_CD = W`는 기타/미확인 범주로 유지한다.

| 컬럼 | 코드값 | 해석 | 상태 | 메모 |
|---|---:|---|---|---|
| SEX_CD | M | 남성 | 확정 | 일반 성별 코드 기준 |
| SEX_CD | F | 여성 | 확정 | 일반 성별 코드 기준 |
| SEX_CD | W | 기타/미확인 | 미확인 | 공식 정의서 확인 전까지 단정 금지 |
| PURPOSE | 0~6 | 이동 목적 코드 | 미확인 | 공식 정의서 기준 매핑 필요 |
| TRANS_GB | 0~7 | 이동수단 코드 | 미확인 | 공식 정의서 기준 매핑 필요 |
| AGE_GRP | 1~12 | 연령대 코드 | 미확인 | 공식 정의서 기준 매핑 필요 |

## 모델링 feature 후보 검증 기준

모델링용 feature table을 만들 때는 기존 parquet 검증 외에 아래 항목을 추가로 확인한다.

| 검증 항목 | 기준 |
|---|---|
| 키 중복 | `base_quarter + admi_cd` 기준 한 행으로 집계되는지 |
| 기간 커버리지 | 2023Q1 ~ 2025Q4 분기별 값이 안정적으로 존재하는지 |
| 결측률 | 주요 feature의 결측률이 과도하지 않은지 |
| 스케일 | CNT 계열은 로그 변환 또는 비율 변환 필요 여부 확인 |
| 중복 변수 | T26/T27처럼 의미가 겹치는 변수는 상관 확인 후 선택 |
| 코드값 | PURPOSE, TRANS_GB는 공식 정의서 전까지 코드값 기준으로만 사용 |